# Практическое задание 2: Улучшение качества LLM классификатора

In [1]:
from IPython.display import clear_output
%pip install pandas mistralai tqdm scikit-learn python-dotenv tabulate tenacity
clear_output()


Файл с данными можно скачать по ссылке: https://disk.yandex.ru/d/ErrpfkgFkdtOIQ.  
Подгружаем данные:

In [2]:
import pandas as pd

data = pd.read_csv("tweets_coronavirus_new.csv")


Настраиваем LLM API:

Переходим на [сайт](https://console.mistral.ai/home). Далее регистрируемся на нем, переходим в API Keys и создаем новый ключ. Его мы будем использовать для авторизации.

Далее, если вы работаете в Google Colab, нужно добавить созданный ключ в Секреты. Для этого откройте соответствующую панель в меню слева и нажмите "Добавить секрет". В поле "Название" введите `MISTRAL_API_KEY`, а в поле "Значение" вставьте ваш ключ. Разрешите доступ из блокнотов. Если же вы работаете в локальной среде, то добавьте созданный ключ в переменные окружения.

### Реализация: локальное подключение

В работе используется локальное окружение и актуальный импорт `from mistralai.client import Mistral`. Ключ загружается из `.env` через `python-dotenv`; значение ключа в вывод не печатается.


In [3]:
from mistralai.client import Mistral
from dotenv import load_dotenv
import os

load_dotenv(".env")
API_KEY = os.environ["MISTRAL_API_KEY"]

llm_client = Mistral(api_key=API_KEY)


Посмотрим на наши данные:

In [4]:
data.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,8041,52993,"Livingston County, NY",18-03-2020,COVID-19 PRICE GOUGING HOTLINE: If you see exc...,Negative
1,22726,67678,"Adelaide, Australia",24-03-2020,URGENT PLEASE SHARE Hearing reports of fatalit...,Negative
2,36557,81509,"McCool, MS",07-04-2020,Taste of Power cuts prices with 50 while donat...,Positive
3,17518,62470,NaN,21-03-2020,Just been to supermarket to get a few things. ...,Negative
4,17470,62422,Morgantown WV,21-03-2020,@jacknicas Right...he had no idea that Covid-1...,Negative


In [5]:
print(data.OriginalTweet[:5].to_markdown())

|    | OriginalTweet                                                                                                                                                                                                                                                                                                                                                                                       |
|---:|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
|  0 | COVID-19 PRICE GOUGING HOTLINE: If you see excessively priced consumer goods and services that are used primarily for personal, family or household purposes to prevent or respond to the COVID-19 viru

In [6]:
data.Sentiment.value_counts()

Sentiment
Negative    59
Positive    41
Name: count, dtype: int64

Наша задача &mdash; предсказывать сентимент твита. Закодируем в числа наш таргет:

In [7]:
def text2label(text) -> int:
    if text in (0, 1):
        return int(text)
    if text in ("Positive", "Extremely Positive"):
        return 1
    if text in ("Negative", "Extremely Negative", "Neutral"):
        return 0
    raise ValueError(f"Неизвестная метка: {text!r}")


In [8]:
data["Sentiment"] = data["Sentiment"].apply(text2label)

In [9]:
data.Sentiment.value_counts()

Sentiment
0    59
1    41
Name: count, dtype: int64

Определим модель, промпты и параметры генерации LLM:

| Настройка | Значение |
|---|---|
| Модель | mistral-large-4 |
| Промпт | базовый system_promt |
| temperature | 0.1 |


In [10]:
MODEL_NAME = "mistral-large-4"  # вы можете попробовать другие модели https://docs.mistral.ai/models

SYSTEM_PROMPT = "Please classify the user's text. Respond with only one word: 'positive' or 'negative'. No other words allowed."

USER_PROMPT = "{}"

SAMPLING_PARAMETERS = {"temperature": 0.1}  # вы можете попробовать поизменять параметры, чтобы добиться лучшего эффекта


Функция для обработки ответа модели. Здесь вы переводите ответ модели в метку класса (0 или 1):

**Изменение обработчика**<br>
Теперь поддерживается и строка, и список текстовых блоков. Перед переводом в 0/1 ответ нормализуется, те убираем пробелы и переносы строк по краям и приводим к нижнему регистру.

In [ ]:
def response2label(response) -> int:
    # print(response)  # debug
    if isinstance(response, list):
        response = "".join(chunk.text for chunk in response if chunk.type == "text")
        
    if not isinstance(response, str):
        raise ValueError(f"Неожиданный ответ: {response!r}")

    label = response.strip().lower()
    if label not in ("positive", "negative"):
        raise ValueError(f"Неожиданная метка: {label!r}")
    return int(label == "positive")


Код для быстрой обработки датасета с помощью LLM:

**Изменение обработки запросов.**<br>
Твиты отправляются последовательно, с паузой 2 секунды после ответа. Для сетевых сбоев я предусмотрел до трёх попыток с паузой 20 секунд. Параметр predictions как раз позволяет продолжить обработку с готовых ответов.


In [12]:
import asyncio
import json
from pathlib import Path
import httpx2
from sklearn.metrics import accuracy_score
from mistralai.client import errors as mistral_errors
from tenacity import retry, stop_after_attempt, wait_fixed, retry_if_exception
from tqdm.auto import tqdm


def transient_error(error):
    return isinstance(error, httpx2.TransportError) or (
        isinstance(error, mistral_errors.SDKError)
        and (error.status_code == 429 or 500 <= error.status_code < 600)
    )


@retry(stop=stop_after_attempt(3), wait=wait_fixed(20),
       retry=retry_if_exception(transient_error), reraise=True)
async def classify_sentiment(text):
    chat_response = await llm_client.chat.complete_async(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.format(text)},
        ],
        **SAMPLING_PARAMETERS
    )
    return response2label(chat_response.choices[0].message.content)


async def classify_batch(texts, delay=2, *, predictions=None):
    if predictions is None:
        predictions = []
    assert len(predictions) <= len(texts)
    remaining_texts = texts.iloc[len(predictions):]
    for text in tqdm(remaining_texts, total=len(texts), initial=len(predictions)):
        predictions.append(await classify_sentiment(text))
        await asyncio.sleep(delay)
    return predictions.copy()


/home/nikita/proj/raif/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Исходный baseline

Фиксируем исходные настройки и классифицируем все 100 твитов. 

In [ ]:
large4_settings = {
    "model": MODEL_NAME,
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt": USER_PROMPT,
    "parameters": SAMPLING_PARAMETERS.copy(),
    "texts": data["OriginalTweet"].tolist(),
}

if "large4_baseline" not in globals():
    large4_baseline = {"settings": large4_settings, "predictions": []}
assert large4_baseline["settings"] == large4_settings, "Изменились данные или настройки"
print("Настройки:", MODEL_NAME, SAMPLING_PARAMETERS, repr(SYSTEM_PROMPT))

y_pred = await classify_batch(data['OriginalTweet'], predictions=large4_baseline["predictions"])

y_true = data["Sentiment"].map(text2label)

acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

Path("large4_baseline_result.json").write_text(
    json.dumps(large4_baseline, ensure_ascii=False, indent=2), encoding="utf-8"
)


Настройки: mistral-large-4 {'temperature': 0.1} "Please classify the user's text. Respond with only one word: 'positive' or 'negative'. No other words allowed."


100%|██████████| 100/100 [12:31<00:00,  7.51s/it]

Accuracy: 0.94


23875

После успешного завершения baseline предсказания и настройки записываются в `large4_baseline_result.json`. При сбое повторный запуск ячейки в том же ядре продолжает обработку через large4_baseline.


## Голосование из трёх повторов

Первый baseline используется как один голос. Выполним ещё два прогона с той же моделью, промптом и температурой. Для каждого твита выберем класс, который предсказали минимум два из трёх прогонов.

In [ ]:
assert "large4_baseline" in globals() and len(large4_baseline["predictions"]) == len(data)

assert large4_baseline["settings"] == {
    "model": MODEL_NAME, "system_prompt": SYSTEM_PROMPT,
    "user_prompt": USER_PROMPT, "parameters": SAMPLING_PARAMETERS,
    "texts": data["OriginalTweet"].tolist(),
}, "Текущие настройки или данные отличаются от первого прогона"

if "large4_repeats" not in globals():
    large4_repeats = {"settings": large4_baseline["settings"], "predictions": [[], []]}
assert large4_repeats["settings"] == large4_baseline["settings"], "Изменились данные или настройки"

for number, predictions in enumerate(large4_repeats["predictions"], start=2):
    print(f"Прогон {number}/3")
    await classify_batch(data["OriginalTweet"], predictions=predictions)
    
    Path("large4_repeats_result.json").write_text(
        json.dumps(large4_repeats, ensure_ascii=False, indent=2), encoding="utf-8"
    )


Прогон 2/3


100%|██████████| 100/100 [11:34<00:00,  6.94s/it]


Прогон 3/3


100%|██████████| 100/100 [12:19<00:00,  7.40s/it]


### Расчёт итоговой метрики


In [ ]:
assert all(len(values) == len(data) for values in large4_repeats["predictions"])
assert large4_repeats["settings"] == large4_baseline["settings"]
assert large4_baseline["settings"]["texts"] == data["OriginalTweet"].tolist()

votes = pd.DataFrame({
    "Baseline": large4_baseline["predictions"],
    "Repeat 2": large4_repeats["predictions"][0],
    "Repeat 3": large4_repeats["predictions"][1],
}, index=data.index)
assert votes.isin([0, 1]).all().all()

y_pred_vote = (votes.sum(axis=1) >= 2).astype(int)
y_true = data["Sentiment"].map(text2label)

for name in votes:
    print(name, "accuracy:", accuracy_score(y_true, votes[name]))
print("Voting accuracy:", accuracy_score(y_true, y_pred_vote))
print("Ошибки голосования:", data.index[y_true != y_pred_vote].tolist())


Baseline accuracy: 0.94
Repeat 2 accuracy: 0.98
Repeat 3 accuracy: 0.95
Voting accuracy: 0.97
Ошибки голосования: [77, 94, 99]


## Результаты

| Вариант | Accuracy | Правильных ответов | Ошибок |
|---|---:|---:|---:|
| Baseline — первый прогон | 0.94 | 94 / 100 | 6 |
| Второй прогон | 0.98 | 98 / 100 | 2 |
| Третий прогон | 0.95 | 95 / 100 | 5 |
| **Большинство трёх прогонов** | **0.97** | **97 / 100** | **3** |

Большинство повысило accuracy относительно первого baseline с 0.94 до 0.97: число ошибок сократилось с шести до трёх, при этом второй отдельный прогон оказался точнее всех.

У голосования остались ошибки на строках 77, 94 и 99, так как твиты включают неоднозначные случаи. Поэтому часть расхождений может быть связана с трактовкой тональности и особенностями разметки.

**Main вывод:** <br>
Для этой выборки сочетание Mistral Large 4, короткого промпта и большинства трёх повторов дало accuracy = 0.97. Усложнение промпта, из моих прошлых экспериментов, только усложняло задачу и путало модель, температура также не дала значимого результата.